# Part 2: Vector store + LLM action plans

Run **`Index.ipynb`** first so `RAG_docs/vector_store/` exists (FAISS index + `rag_parents.json`).

**This notebook** loads the **saved vector store only** (no re-reading `RAG_docs/knowledge/`) and writes JSON under `RAG_docs/action_plans/` (including optional rerank comparison reports).

**Run order:** imports → **load data** (builds `samples_for_actions`) → **definitions** (retrieval + pipeline; edit `_PIPELINE_*` flags there to control how much RAG text / full prompt is printed) → **driver** cells (preload → configuration → execute).

## Retrieval improvements (what runs before the LLM)

Retrieval is a multi-stage ranking pipeline:

- **Multi-query support**: `QUERY_STRATEGY` can be a single strategy or a list of strategies.
- **Per-query retrieval**: fetch up to **100** nearest child chunks per query (after index oversample + balance), then MMR/rerank on **100** candidates.
- **Merge + dedupe**: merge candidates and remove duplicates (prefers `(parent_id, child_index)` identity).
- **MMR**: diversify results to reduce redundant chunks while keeping relevance.
- **Reranking**: `rerank_mode` can be `none`, `crossencoder`, `colbert`, or `crossencoder_colbert`. With **`RUN_RERANK_ABLATION = True`** (configuration driver cell), compare modes and save `rerank_comparison_sample_*.json`.
- **Child → parent expansion**: expand only the top-ranked child chunks into parent documents.
- **Final context**: pass the **top 10** ranked parent sections (full text) into the LLM prompt.

**Shared helpers:** `utils/rag_utils.py` (FAISS load/save, prediction/config loaders, etc.). Retrieval + action planning logic lives in this notebook.

## Evaluation goals (rerank / retrieval metrics)

**Decision:** optimize **both** (they answer different questions).

1. **Downstream action-plan quality** — primary product goal: grounded JSON plans (threat level, priorities, actions, `knowledge_sources_used`) that match operational needs. Prefer human rubric, LLM-as-judge, or schema/citation checks over keyword overlap on chunks alone.
2. **Query–chunk relevance** — leading indicator: same embedder as the FAISS index, e.g. mean cosine(query, top-*k* chunk texts) per `rerank_mode`. Helps diagnose retrieval without replacing end-to-end evaluation.

Use `rag_context_rerank_keyword_scores.csv` only as a **diagnostic** (label/action/tier word overlap), not as the main leaderboard. Optional tooling: `utils/rerank_metrics.py` and the **Rerank report metrics** cell at the end of this notebook.

## Inline Parameter Annotations (Plan)

These are the highest-impact parameters for retrieval, reranking, and action-plan output quality.

- `VECTOR_STORE_DIR` / `PREDICTIONS_DIR` / `RESULTS_DIR`: I/O contract for loading FAISS + predictions and saving action plans.
- `_INDEX_EMBED_MODEL` from manifest: keeps retrieval diagnostics tied to the exact embedding model used to build the index.
- `_PER_QUERY_RETRIEVE_K`: candidate width per query before reranking; larger improves recall but increases latency.
- `_DEFAULT_MMR_K`: pool size considered by MMR; controls redundancy suppression strength.
- `_DEFAULT_RERANK_K`: number of candidates scored by rerankers; larger improves ranking headroom with higher compute.
- `_DEFAULT_FINAL_SECTIONS`: final context budget passed to LLM; more sections add recall but can dilute prompt focus.
- `oversample_factor` in `retrieve_child_chunks_for_query(...)`: initial FAISS overfetch multiplier to improve post-filter quality.
- `balance_by_source_file`: prevents one document source from dominating retrieved context.
- `rerank_mode` (`none`, `crossencoder`, `colbert`, `crossencoder_colbert`): ranking strategy choice; primary quality/latency tradeoff.
- `RUN_RERANK_ABLATION`: when enabled, executes all rerank modes and writes comparison reports for offline analysis.
- `ensure_msvc_for_torch_jit()` + `apply_colbert_segmented_maxsim_patch()`: Windows-specific ColBERT compile safeguards.
- LLM prompt flags (`_PIPELINE_*` print/debug controls): tune observability and output verbosity while validating retrieval behavior.
- `DEFAULT_EXPORT_RAG_TOP_N` (via utils): cap for serialized context in output JSON; affects report size and auditability.


In [1]:
import os
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from openai import OpenAI

from utils.project_env import load_project_environment

load_project_environment()

from utils.rag_utils import (
    balance_vector_hits_by_source_file,
    get_dominant_party_info,
    load_attack_and_agentic,
    load_parent_store,
    load_predictions,
    load_vector_store,
    read_manifest,
    save_action_plan,
    save_rerank_comparison_report,
)

VECTOR_STORE_DIR = Path("RAG_docs/vector_store")
PREDICTIONS_DIR = Path("RAG_docs/predictions")
RESULTS_DIR = Path("RAG_docs/action_plans")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# From FAISS manifest; persisted on rerank comparison JSON for offline metrics.
_INDEX_EMBED_MODEL = read_manifest(VECTOR_STORE_DIR).get("embed_model")

In [2]:
vector_store = load_vector_store(VECTOR_STORE_DIR)
predictions_data = load_predictions(PREDICTIONS_DIR, verbose=True)
attack_actions_data, agentic_features_data = load_attack_and_agentic(verbose=True)

n_attack = (
    len(attack_actions_data.get("attacks", {}))
    if isinstance(attack_actions_data, dict)
    else 0
)
has_agentic = isinstance(agentic_features_data, dict) and any(
    t in agentic_features_data for t in ("RAN", "Edge", "Core")
)
print("-" * 60)
print(
    f"Predictions: {len(predictions_data)} | "
    f"Vector store: {'loaded' if vector_store is not None else 'missing'}"
)
print(f"Attack types in attack_options: {n_attack}")
print(f"Agentic features config: {'yes' if has_agentic else 'no'}")
print("-" * 60)


def _include_sample_for_action_plan(sample: Dict[str, Any]) -> bool:
    """Include rows with a non-empty predicted label (including BENIGN)."""
    pl = str(sample.get("predicted_label") or "").strip().upper()
    return bool(pl)


samples_for_actions = [s for s in predictions_data if _include_sample_for_action_plan(s)]
n_empty = len(predictions_data) - len(samples_for_actions)
print(
    f"Samples queued for action planning: {len(samples_for_actions)}"
    + (f" ({n_empty} row(s) with empty predicted_label skipped)." if n_empty else ".")
)


D:\Projects\VFL-RAG\utils\rag_utils.py:213: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return SentenceTransformerEmbeddings(


Found 1 prediction file(s) in RAG_docs\predictions
  Loading: sample_predictions.json
    Loaded 9 prediction(s) from sample_predictions.json
Total: 9 prediction(s)
Loaded attack actions from attack_options.json
Loaded agentic features from agentic_features.json
------------------------------------------------------------
Predictions: 9 | Vector store: loaded
Attack types in attack_options: 9
Agentic features config: no
------------------------------------------------------------
Samples queued for action planning: 9.


In [3]:
# RAG retrieval + LLM query/prompts/pipeline (search lives here next to query construction)

_RAG_PARENTS = load_parent_store(VECTOR_STORE_DIR)


_PER_QUERY_RETRIEVE_K = 100

# Ranking width vs LLM context: retrieve/MMR/rerank on up to 100; only _DEFAULT_FINAL_SECTIONS go to the prompt.
# Defaults for multi-stage ranking pipeline
_DEFAULT_FINAL_SECTIONS = 10
_DEFAULT_MMR_K = 100
_DEFAULT_RERANK_K = 100


def _chunk_key(d: Dict[str, Any]) -> Tuple[Any, ...]:
    """Stable identity key for a child chunk."""
    pid = d.get("parent_id")
    cix = d.get("child_index")
    if pid is not None and cix is not None:
        return ("pc", str(pid), int(cix))
    return (
        "legacy",
        str(d.get("source_file") or "").strip(),
        str(d.get("title") or "").strip(),
        str(d.get("chunk_text") or "").strip()[:120],
    )


def _as_score_from_dist(dist: Any) -> float:
    d = float(dist) if dist is not None else 0.0
    return 1.0 / (1.0 + max(d, 0.0))


def retrieve_child_chunks_for_query(
    vector_store: Any,
    query: str,
    *,
    top_k: int = _PER_QUERY_RETRIEVE_K,
    oversample_factor: int = 6,
    balance_by_source_file: bool = True,
) -> List[Dict[str, Any]]:
    """Retrieve child chunks only (NO parent expansion)."""
    if not vector_store or not query:
        return []

    fetch_k = min(300, max(int(top_k), int(top_k) * int(oversample_factor)))
    vector_results = vector_store.similarity_search_with_score(query, k=fetch_k)

    if balance_by_source_file:
        vector_results = balance_vector_hits_by_source_file(vector_results, int(top_k))
    else:
        vector_results = vector_results[: int(top_k)]

    out: List[Dict[str, Any]] = []
    for doc, dist in vector_results:
        meta = doc.metadata or {}
        title = meta.get("retrieval_title") or meta.get("title", "Unknown")
        src = meta.get("source_file", "")
        pid = meta.get("parent_id")
        cix = meta.get("child_index")
        chunk_text = doc.page_content or meta.get("text", "") or ""

        out.append(
            {
                "title": title,
                "source_file": src,
                "parent_id": pid,
                "child_index": cix,
                "chunk_text": chunk_text,
                "vector_score": _as_score_from_dist(dist),
            }
        )

    return out


def merge_and_dedupe_child_chunks(
    results_by_query: List[List[Dict[str, Any]]],
) -> List[Dict[str, Any]]:
    """Merge child-chunk sets; dedupe by (parent_id, child_index) when available."""
    best: Dict[Tuple[Any, ...], Dict[str, Any]] = {}

    for results in results_by_query:
        for d in results:
            k = _chunk_key(d)
            prev = best.get(k)
            if not prev or float(d.get("vector_score", 0.0) or 0.0) > float(prev.get("vector_score", 0.0) or 0.0):
                best[k] = d

    merged = list(best.values())
    merged.sort(key=lambda x: float(x.get("vector_score", 0.0) or 0.0), reverse=True)
    return merged


def _cosine(a: List[float], b: List[float]) -> float:
    import math

    if not a or not b or len(a) != len(b):
        return 0.0
    da = sum(x * x for x in a)
    db = sum(x * x for x in b)
    if da <= 0.0 or db <= 0.0:
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    return float(dot) / float(math.sqrt(da) * math.sqrt(db))


def _embed_query(vector_store: Any, query: str) -> List[float]:
    # LangChain FAISS typically exposes `embedding_function`.
    ef = getattr(vector_store, "embedding_function", None)
    if ef is None:
        raise ValueError("vector_store.embedding_function missing; cannot run MMR")
    if hasattr(ef, "embed_query"):
        return list(ef.embed_query(query))
    # Fallback for older wrappers
    if callable(ef):
        return list(ef(query))
    raise ValueError("Unsupported embedding function")


def _embed_texts(vector_store: Any, texts: List[str]) -> List[List[float]]:
    ef = getattr(vector_store, "embedding_function", None)
    if ef is None:
        raise ValueError("vector_store.embedding_function missing; cannot run MMR")
    if hasattr(ef, "embed_documents"):
        return [list(v) for v in ef.embed_documents(texts)]
    # Slow fallback: embed one-by-one via embed_query
    if hasattr(ef, "embed_query"):
        return [list(ef.embed_query(t)) for t in texts]
    raise ValueError("Unsupported embedding function")


def mmr_select(
    vector_store: Any,
    query: str,
    candidates: List[Dict[str, Any]],
    *,
    k: int = _DEFAULT_MMR_K,
    lambda_mult: float = 0.5,
) -> List[Dict[str, Any]]:
    """MMR over candidate chunk embeddings."""
    if not candidates:
        return []

    k = max(1, min(int(k), len(candidates)))

    texts = [str(c.get("chunk_text") or "") for c in candidates]
    qv = _embed_query(vector_store, query)
    dvs = _embed_texts(vector_store, texts)

    # Precompute query similarities
    sim_q = [_cosine(qv, dv) for dv in dvs]

    selected: List[int] = []
    remaining = set(range(len(candidates)))

    # Start from best by query similarity
    first = max(remaining, key=lambda i: sim_q[i])
    selected.append(first)
    remaining.remove(first)

    while remaining and len(selected) < k:
        def mmr_score(i: int) -> float:
            # max similarity to already-selected docs
            max_sim = max(_cosine(dvs[i], dvs[j]) for j in selected) if selected else 0.0
            return float(lambda_mult) * float(sim_q[i]) - (1.0 - float(lambda_mult)) * float(max_sim)

        nxt = max(remaining, key=mmr_score)
        selected.append(nxt)
        remaining.remove(nxt)

    return [candidates[i] for i in selected]


_CROSSENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
_COLBERT_MODEL_NAME = "colbert-ir/colbertv2.0"

_CROSS_ENCODER = None
_COLBERT = None
_RERANKERS_CONFIRMED = False


def ensure_rerankers_loaded(
    *,
    crossencoder_model: str = _CROSSENCODER_MODEL_NAME,
    colbert_model: str = _COLBERT_MODEL_NAME,
) -> Tuple[Any, Any]:
    """Mandatory reranker loader.

    Raises a clear error if dependencies/models are missing.
    """
    global _CROSS_ENCODER, _COLBERT, _RERANKERS_CONFIRMED

    if _CROSS_ENCODER is None:
        try:
            from sentence_transformers import CrossEncoder  # type: ignore
        except Exception as e:
            raise ImportError(
                "CrossEncoder reranker is REQUIRED. Install: pip install sentence-transformers"
            ) from e
        from utils.compute_device import cross_encoder_device_string

        _CROSS_ENCODER = CrossEncoder(
            crossencoder_model,
            device=cross_encoder_device_string(),
        )

    if _COLBERT is None:
        from utils.msvc_jit_env import ensure_msvc_for_torch_jit

        ensure_msvc_for_torch_jit()
        from utils.colbert_windows_cpp import apply_colbert_segmented_maxsim_patch

        apply_colbert_segmented_maxsim_patch()
        try:
            import utils.langchain_ragatouille_shim  # noqa: F401 — LC 1.x vs RAGatouille import paths
            from ragatouille import RAGPretrainedModel  # type: ignore
        except Exception as e:
            raise ImportError(
                "ColBERT reranker is REQUIRED. Install: pip install ragatouille"
            ) from e
        _COLBERT = RAGPretrainedModel.from_pretrained(colbert_model)

    if not _RERANKERS_CONFIRMED:
        print(
            "Rerankers loaded OK: "
            f"CrossEncoder='{crossencoder_model}', ColBERT='{colbert_model}'"
        )
        _RERANKERS_CONFIRMED = True

    return (_CROSS_ENCODER, _COLBERT)


def crossencoder_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[float]:
    """Cross-encoder rerank scores; mandatory (no fallback)."""
    ce, _ = ensure_rerankers_loaded()
    pairs = [(query, str(c.get("chunk_text") or "")) for c in candidates]
    scores = ce.predict(pairs)
    return [float(s) for s in scores]


def colbert_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[float]:
    """ColBERT rerank scores; mandatory (no fallback)."""
    _, cb = ensure_rerankers_loaded()
    docs = [str(c.get("chunk_text") or "") for c in candidates]
    scored = cb.rerank(query=query, documents=docs, k=len(docs))

    # Keep ordering; ragatouille returns dicts containing the raw `document` string.
    score_map = {str(x.get("document")): float(x.get("score", 0.0) or 0.0) for x in scored}
    return [float(score_map.get(d, 0.0)) for d in docs]


def rerank_crossencoder_plus_colbert(
    query: str,
    candidates: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Attach rerank scores and sort descending."""
    ce = crossencoder_rerank(query, candidates)
    cb = colbert_rerank(query, candidates)

    out: List[Dict[str, Any]] = []
    for c, s_ce, s_cb in zip(candidates, ce, cb):
        d = dict(c)
        d["crossencoder_score"] = float(s_ce)
        d["colbert_score"] = float(s_cb)
        # Simple combination; you can tune weights later.
        d["rerank_score"] = float(s_ce) + float(s_cb)
        out.append(d)

    out.sort(key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True)
    return out


def expand_parent_sections(
    ranked_children: List[Dict[str, Any]],
    *,
    top_sections: int = _DEFAULT_FINAL_SECTIONS,
    max_parent_chars: int = 12000,
) -> List[Dict[str, Any]]:
    """Expand top-ranked child chunks into parent contextual sections.

    Removes duplicates by keeping the most relevant section when the same
    (parent_id) or (source_file,title) repeats.
    """
    if not ranked_children:
        return []

    # Build candidate sections first (may include duplicates)
    candidates: List[Dict[str, Any]] = []

    for c in ranked_children:
        score = float(c.get("rerank_score", c.get("vector_score", 0.0) or 0.0))
        pid = c.get("parent_id")

        if pid is None:
            candidates.append(
                {
                    "title": c.get("title", "Unknown"),
                    "source_file": c.get("source_file", ""),
                    "text": str(c.get("chunk_text") or ""),
                    "score": score,
                    "child_index": c.get("child_index"),
                }
            )
            continue

        pid_s = str(pid)
        parent = _RAG_PARENTS.get(pid_s) or _RAG_PARENTS.get(pid) or {}
        ptext = str(parent.get("text") or "")
        if max_parent_chars and len(ptext) > int(max_parent_chars):
            ptext = ptext[: int(max_parent_chars)] + "\n\n[parent truncated]"

        title = parent.get("retrieval_title") or c.get("title") or "Unknown"
        candidates.append(
            {
                "title": title,
                "source_file": c.get("source_file", ""),
                "text": f"{title}\n\n{ptext}" if ptext else str(c.get("chunk_text") or ""),
                "score": score,
                "parent_id": pid_s,
                "child_index": c.get("child_index"),
            }
        )

    # Dedupe: prefer parent_id when present, otherwise (source_file,title)
    best: Dict[Tuple[Any, ...], Dict[str, Any]] = {}
    for s in candidates:
        if s.get("parent_id"):
            k: Tuple[Any, ...] = ("parent", str(s.get("parent_id")))
        else:
            k = ("doc", str(s.get("source_file") or "").strip(), str(s.get("title") or "").strip())

        prev = best.get(k)
        if not prev or float(s.get("score", 0.0) or 0.0) > float(prev.get("score", 0.0) or 0.0):
            best[k] = s

    sections = list(best.values())
    sections.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return sections[: int(top_sections)]


def retrieve_rag_context_multi(
    vector_store: Any,
    queries: List[str],
    *,
    top_k: int = _DEFAULT_FINAL_SECTIONS,
    oversample_factor: int = 6,
    balance_by_source_file: bool = True,
    mmr_k: int = _DEFAULT_MMR_K,
    rerank_k: int = _DEFAULT_RERANK_K,
    lambda_mult: float = 0.5,
    max_parent_chars: int = 12000,
    rerank_mode: str = "crossencoder_colbert",
) -> Tuple[List[Dict[str, Any]], List[List[Dict[str, Any]]]]:
    """Merge + dedupe → MMR → rerank (see ``rerank_mode``) → parent expansion → top sections.

    ``rerank_mode``: ``none`` | ``crossencoder`` | ``colbert`` | ``crossencoder_colbert``.
    """
    qs = [" ".join((q or "").split()) for q in (queries or [])]
    qs = [q for q in qs if q]
    if not qs:
        return ([], [])

    per_query: List[List[Dict[str, Any]]] = []
    for q in qs:
        per_query.append(
            retrieve_child_chunks_for_query(
                vector_store,
                q,
                top_k=int(_PER_QUERY_RETRIEVE_K),
                oversample_factor=oversample_factor,
                balance_by_source_file=balance_by_source_file,
            )
        )

    merged = merge_and_dedupe_child_chunks(per_query)

    anchor_query = qs[0]

    mmr_pool = mmr_select(
        vector_store,
        anchor_query,
        merged,
        k=int(mmr_k),
        lambda_mult=float(lambda_mult),
    )

    rm = (rerank_mode or "crossencoder_colbert").strip().lower()
    if rm == "none":
        reranked: List[Dict[str, Any]] = []
        for c in mmr_pool:
            d = dict(c)
            vs = float(d.get("vector_score", 0.0) or 0.0)
            d["rerank_score"] = vs
            d["ranking_method"] = "vector_only"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    elif rm == "crossencoder":
        ce_scores = crossencoder_rerank(anchor_query, mmr_pool)
        reranked = []
        for c, s_ce in zip(mmr_pool, ce_scores):
            d = dict(c)
            d["crossencoder_score"] = float(s_ce)
            d["rerank_score"] = float(s_ce)
            d["ranking_method"] = "crossencoder"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    elif rm == "colbert":
        cb_scores = colbert_rerank(anchor_query, mmr_pool)
        reranked = []
        for c, s_cb in zip(mmr_pool, cb_scores):
            d = dict(c)
            d["colbert_score"] = float(s_cb)
            d["rerank_score"] = float(s_cb)
            d["ranking_method"] = "colbert"
            reranked.append(d)
        reranked.sort(
            key=lambda x: float(x.get("rerank_score", 0.0) or 0.0), reverse=True
        )
    else:
        reranked = rerank_crossencoder_plus_colbert(anchor_query, mmr_pool)
        for d in reranked:
            d["ranking_method"] = "crossencoder_colbert"

    top_children = reranked[: int(rerank_k)]
    final_sections = expand_parent_sections(
        top_children,
        top_sections=int(top_k),
        max_parent_chars=int(max_parent_chars),
    )

    final_sections.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return (final_sections[: int(top_k)], per_query)


def summarize_rag_docs(
    docs: List[Dict[str, Any]],
    *,
    max_docs: int = 5,
    snippet_chars: int = 220,
) -> List[Dict[str, Any]]:
    """Compact summary objects for printing/debug.

    Supports both shapes:
    - final sections: {title, text, score, source_file}
    - child chunks:   {title, chunk_text, vector_score, source_file}
    """
    out: List[Dict[str, Any]] = []
    for d in (docs or [])[: int(max_docs)]:
        txt = str(d.get("text") or d.get("chunk_text") or "")
        snippet = txt[: int(snippet_chars)]
        if len(txt) > int(snippet_chars):
            snippet += "..."

        score = d.get("score")
        if score is None:
            score = d.get("rerank_score")
        if score is None:
            score = d.get("vector_score")

        out.append(
            {
                "title": d.get("title", "Unknown"),
                "source_file": d.get("source_file", ""),
                "score": float(score or 0.0),
                "snippet": " ".join(snippet.split()),
            }
        )
    return out


def _top_features_for_tier(
    sample: Dict[str, Any],
    tier: str,
    *,
    top_n: int = 3,
    score_key: str = "pct_contribution",
) -> List[str]:
    """Deterministic top-N feature names for a tier (RAN/Edge/Core)."""
    shap_expl = sample.get("shap_explanation", {}) or {}
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    feats = feat_contribs.get(tier, {}) or {}

    scored: List[Tuple[str, float]] = []
    if isinstance(feats, dict):
        for feat_name, meta in feats.items():
            if not isinstance(meta, dict):
                continue
            score = float(meta.get(score_key, 0.0) or 0.0)
            if score > 0.0:
                scored.append((feat_name, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [name for name, _ in scored[:top_n]]


def extract_sample_summary(sample: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize prediction fields used by query builders (single source of truth)."""
    label = (sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN").strip().upper()
    confidence = float(sample.get("confidence", 0.0) or 0.0)

    shap_expl = sample.get("shap_explanation", {}) or {}
    dominant_agent = (shap_expl.get("dominant_agent") or "").strip()
    dominant_tier = dominant_agent if dominant_agent in ("RAN", "Edge", "Core") else "Unknown"
    dominant_pct = float(shap_expl.get("dominant_contribution_pct", 0.0) or 0.0) * 100.0

    top_features = {
        "RAN": _top_features_for_tier(sample, "RAN", top_n=3),
        "Edge": _top_features_for_tier(sample, "Edge", top_n=3),
        "Core": _top_features_for_tier(sample, "Core", top_n=3),
    }

    return {
        "label": label,
        "confidence": confidence,
        "dominant_tier": dominant_tier,
        "dominant_pct": dominant_pct,
        "top_features": top_features,
    }


_TEMPLATE_QUERY = (
    "Find mitigation actions, detection steps, and response playbooks for a {label} attack in a telecom network. "
    "Prioritize the {dominant_tier} tier (dominant contribution ~{dominant_pct:.0f}%). "
    "Use these top indicators as keywords: "
    "RAN: {ran_feats}. Edge: {edge_feats}. Core: {core_feats}. "
    "Focus on controls appropriate for confidence {confidence:.0%}."
)


def build_template_rag_query(sample: Dict[str, Any]) -> str:
    """Deterministic template query used for vector retrieval (NO LLM).

    Query options supported in this notebook:
    1) Template-based (deterministic)            -> `build_template_rag_query(sample)`
    2) "BERT-based" style rephrase (optional)     -> `rephrase_template_query_deterministic(template_query)`
       (Note: implemented as deterministic rules; no model downloads.)
    3) LLM-based query variants (optional)       -> `build_llm_rag_queries(sample)`
       (Returns two variants derived from the template + top features in prediction JSON.)

    For RAG retrieval, we intentionally use **only option (1)** for determinism.
    """
    s = extract_sample_summary(sample)

    def fmt(feats: List[str]) -> str:
        return ", ".join(feats) if feats else "no strong features"

    return _TEMPLATE_QUERY.format(
        label=s["label"],
        dominant_tier=s["dominant_tier"],
        dominant_pct=s["dominant_pct"],
        confidence=s["confidence"],
        ran_feats=fmt(s["top_features"]["RAN"]),
        edge_feats=fmt(s["top_features"]["Edge"]),
        core_feats=fmt(s["top_features"]["Core"]),
    )


def build_llm_rag_queries(sample: Dict[str, Any]) -> Tuple[str, str]:
    """Optional: return (concise_query, expanded_query) using an LLM.

    Not used for retrieval by default.
    """
    base = build_template_rag_query(sample)
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return (base, base)

    client = OpenAI(api_key=api_key)
    prompt = (
        "Rewrite the following retrieval query into two variants for vector search.\n\n"
        "Variant A: concise (1 sentence, keep feature tokens).\n"
        "Variant B: expanded (2-3 sentences, add synonyms, keep feature tokens).\n\n"
        "Return JSON only: {\"concise\": \"...\", \"expanded\": \"...\"}.\n\n"
        f"QUERY:\n{base}"
    )

    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "Return only valid JSON."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=250,
    )

    txt = (resp.choices[0].message.content or "").strip()
    try:
        start = txt.find("{")
        end = txt.rfind("}") + 1
        obj = json.loads(txt[start:end]) if start >= 0 and end > start else {}
        concise = str(obj.get("concise") or base).strip()
        expanded = str(obj.get("expanded") or base).strip()
        return (concise, expanded)
    except Exception:
        return (base, base)


def rephrase_template_query_deterministic(template_query: str) -> str:
    """Deterministic query rephrase (no models / no downloads).

    Not used for retrieval by default.
    """
    q = (template_query or "").strip()
    if not q:
        return q

    # Simple, stable synonym swaps + light restructuring.
    swaps = [
        ("Find mitigation actions, detection steps, and response playbooks for a ", "Retrieve mitigation and detection guidance for "),
        ("telecom network", "telecommunications network"),
        ("Prioritize the ", "Focus on the "),
        ("Use these top indicators as keywords:", "Key indicators:"),
        ("Focus on controls appropriate for confidence", "Tailor controls to confidence"),
    ]

    for a, b in swaps:
        q = q.replace(a, b)

    # Ensure a compact single-line output for embedding/search.
    q = " ".join(q.split())
    return q


def create_prompt(
    sample_data: Dict[str, Any],
    rag_results: List[Dict[str, Any]],
    attack_actions_data: Optional[Dict[str, Any]] = None,
    agentic_features_data: Optional[Dict[str, Any]] = None,
) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample_data)

    # IMPORTANT: Do NOT pass the full prediction/SHAP payload to the LLM.
    # Only pass a compact evidence summary: top-3 features per tier + dominant tier.
    s = extract_sample_summary(sample_data)
    condensed_evidence = {
        "predicted_label": str(s.get("label") or predicted_label),
        "confidence": float(s.get("confidence", confidence) or 0.0),
        "dominant_tier": s.get("dominant_tier", dominant_tier),
        "dominant_contribution_pct": float(s.get("dominant_pct", dominant_pct) or 0.0),
        "top_features_by_tier": s.get("top_features", {}),
    }

    attack_actions_context = ""
    if attack_actions_data and "attacks" in attack_actions_data:
        attack_type = predicted_label.upper()
        if attack_type in attack_actions_data["attacks"]:
            recommended_actions = attack_actions_data["attacks"][attack_type]
            attack_actions_context = (
                f"\n\nAttack-Specific Recommended Actions (from attack_options.json):\n"
                f"For {predicted_label} attack, recommended actions: {', '.join(recommended_actions)}\n"
            )
        else:
            attack_actions_context = (
                f"\n\nAttack-Specific Actions: No specific recommendations for {predicted_label}.\n"
            )

    agentic_context = ""
    if agentic_features_data:
        agentic_context = (
            "\n\nAgentic Features and Actions by Network Tier (from agentic_features.json):\n"
            "Use the condensed evidence (top features by tier) to GATE which tier gets priority actions.\n"
        )
        tf = condensed_evidence.get("top_features_by_tier", {}) or {}
        for tier in ["RAN", "Edge", "Core"]:
            if tier in agentic_features_data:
                tier_data = agentic_features_data[tier]
                actions = tier_data.get("actions", [])
                tier_feats = tf.get(tier, []) if isinstance(tf, dict) else []
                agentic_context += f"\n{tier} Network Tier:\n"
                agentic_context += f"  - Top evidence features (top 3): {', '.join(tier_feats) if tier_feats else 'none'}\n"
                agentic_context += f"  - Allowed tier actions: {', '.join(actions)}\n"

    # RAG context: shared helper so the notebook can print the same text the model sees.
    rag_context = build_rag_context_text(rag_results)

    network_tier_info = ""
    if dominant_tier:
        network_tier_info = f"\n- Dominant network tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)"
        if dominant_party:
            network_tier_info += f"\n- Dominant party: {dominant_party}"

    return f"""You are a cybersecurity decision-making agent specialized in attack response orchestration.
        Your role is NOT to invent mitigations, but to SELECT and ASSIGN actions from a predefined
        action set using explainability signals and agentic features.

        =====================
        INPUT CONTEXT
        =====================

        Prediction summary:
        - sample_id: {sample_id}
        - predicted_label: {predicted_label}
        - confidence: {confidence:.1%}

        Explainability & agentic evidence:
        - Party-level contributions and dominance:
        {network_tier_info}

        - Condensed feature evidence (top 3 per tier):
        {json.dumps(condensed_evidence, indent=2)}

        Allowed actions (STRICT CONSTRAINT):
        {attack_actions_context}
        • Only actions listed above are allowed.
        • Do NOT invent, rename, generalize, or merge actions.

        Agentic decision signals:
        {agentic_context}

        Retrieved knowledge (RAG – optional support, may be empty):
        {rag_context}

        =====================
        TASK
        =====================

        Using ONLY the information above, generate an agent-ready action plan that:

        1) Interprets how {predicted_label} manifests across network tiers (RAN, Edge, Core).
        2) Identifies which evidence party MUST trigger mitigation first.
        3) Selects actions ONLY from the provided Allowed actions list.
        4) Assigns each action to the MOST appropriate executing party and network tier.
        5) Provides explicit, evidence-grounded reasoning for EACH action.
        6) Adapts aggressiveness and execution priority based on confidence ({confidence:.1%}).
        7) If no allowed action is suitable, return empty action lists.

        =====================
        OUTPUT FORMAT (STRICT)
        =====================

        Return a VALID JSON object ONLY:

        {{
          "threat_level": "Critical|High|Medium|Low",

          "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
          ],

          "primary_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken ",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],

          "supporting_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "why this action supports or complements the primary action"
            }}
          ],
          "overall_reasoning": "Concise summary explaining party prioritization, tier ordering, and action selection logic",
          "execution_priority": "Immediate|High|Standard|Low",
          "knowledge_sources_used": [
            "allowed_actions_context",
            "attack_actions_context"
          ]
        }}

        =====================
        HARD RULES (DO NOT VIOLATE)
        =====================

        - Do NOT output text outside the JSON.
        - Do NOT generate actions not listed in Allowed actions.
        - The "all_actions" list MUST be the union of primary_actions and supporting_actions.
        - Do NOT alter action or party names.
        - Every action MUST include explicit reasoning tied to evidence or agentic rules.
        - Prefer dominant party and tier for primary actions unless contradicted by evidence.
        - If RAG context is empty, rely ONLY on explainability and agentic context.
        """


def create_prompt_without_RAG(sample_data: Dict[str, Any]) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)

    return f"""You are a cybersecurity expert responsible for selecting mitigation actions
      based on attack predictions and domain knowledge.

      Your objective is to decide WHAT actions should be taken and WHY,
      strictly using a predefined set of allowed actions.

      =====================
      INPUT
      =====================

      Prediction summary:
      - sample_id: {sample_id}
      - predicted_label: {predicted_label}
      - confidence: {confidence:.1%}

      =====================
      TASK
      =====================

      Using ONLY the information above:

      1) Assess the severity of the predicted attack ({predicted_label})
        and assign an appropriate threat level.
      2) Select all relevant mitigation actions within 2 or 3 words
      3) Ensure selected actions are appropriate for the predicted attack type.
      4) Adapt action selection and urgency based on confidence ({confidence:.1%}).
      5) If no action is applicable, return an empty action list.

      =====================
      OUTPUT (STRICT JSON ONLY)
      =====================

      {{
        "threat_level": "Critical|High|Medium|Low",
        "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
        ],
        "primary_actions": [
            {{
              "action": "any action name  to be taken",
              "network_tier": "RAN|Edge|Core",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],
        "overall_reasoning": "Brief explanation describing why these actions were selected and how confidence influenced the decision",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": [
          "publicly_available_online_sources"
        ]
      }}

      =====================
      HARD RULES
      =====================

      - Do NOT output text outside the JSON.
      - Do NOT invent, rename, or generalize actions.
      - Use short, standard mitigation action names (2–3 words).
      - If no action applies, return empty lists for `all_actions` and `primary_actions`.
      - Keep reasoning concise and decision-focused.
      """


def call_llm_api(prompt: str) -> Optional[Dict[str, Any]]:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {
                "role": "system",
                "content": "You are a cybersecurity expert. Return only valid JSON.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
    )

    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find("{")
        end = response_text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")

    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": [],
    }


def _normalize_strategy_to_keys(strategy: Any) -> List[str]:
    """Normalize a strategy value into a list of strategy keys."""
    if isinstance(strategy, list):
        return [str(x).strip().lower() for x in strategy if str(x).strip()]
    s = str(strategy or "").strip().lower()
    return [s] if s else ["template"]


def _vprint(msg: str = "") -> None:
    print(msg)


# --- Notebook observability (stdout when you run the pipeline driver) ---
_PIPELINE_SHOW_UNUSED_QUERY_VARIANTS = False
_PIPELINE_PRINT_PER_QUERY_CHUNKS = True
_PIPELINE_PRINT_MERGED_CHUNK_SUMMARY = True
_PIPELINE_PRINT_RAG_CONTEXT_VERBATIM = True
_PIPELINE_PRINT_FULL_LLM_PROMPT = True
_PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE = False
# Compact driver output: attack type, top-N source|title per rerank mode, full LLM JSON.
_PIPELINE_COMPACT_STDOUT = False
_PIPELINE_COMPACT_TOP_SOURCES = 10


def build_rag_context_text(
    rag_results: List[Dict[str, Any]],
    *,
    max_sections: int = 10,
) -> str:
    "Exact Knowledge Base block embedded in the LLM user prompt (for notebook inspection)."
    top = (rag_results or [])[: int(max_sections)]
    if not top:
        return "\n\nKnowledge Base: No relevant documents found from RAG search."
    parts = ["\n\nKnowledge Base Context (from RAG search):\n"]
    for idx, result in enumerate(top, 1):
        title = result.get("title", "Unknown")
        parts.append(f"\n[{idx}] {title}\n")
        parts.append(f"{result.get('text', '')}\n")
    return "".join(parts)


def _print_pipeline_prompt_artifacts(
    rag_results: List[Dict[str, Any]],
    prompt: str,
    *,
    header_extra: str = "",
    ablation_mode_index: Optional[int] = None,
    ablation_modes_total: Optional[int] = None,
) -> None:
    is_ablation = (
        ablation_mode_index is not None
        and ablation_modes_total is not None
        and int(ablation_modes_total) > 1
    )
    print_full = bool(_PIPELINE_PRINT_FULL_LLM_PROMPT)
    if is_ablation and not _PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE:
        print_full = print_full and int(ablation_mode_index) == 0

    if _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM:
        _vprint("\n" + "=" * 80)
        _vprint("RAG CONTEXT (verbatim block inside the LLM user message)" + header_extra)
        _vprint("=" * 80)
        _vprint(build_rag_context_text(rag_results))

    if print_full:
        _vprint("\n" + "=" * 80)
        _vprint(f"LLM SYSTEM MESSAGE (API; see call_llm_api){header_extra}")
        _vprint("=" * 80)
        _vprint("You are a cybersecurity expert. Return only valid JSON.")
        _vprint("\n" + "=" * 80)
        _vprint(f"FULL LLM USER PROMPT{header_extra} — {len(prompt)} characters")
        _vprint("=" * 80)
        _vprint(prompt)
    elif is_ablation and _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM:
        _vprint(
            f"\n(Full user prompt omitted for this ablation mode; "
            f"length {len(prompt)} chars. "
            f"Set _PIPELINE_PRINT_FULL_PROMPT_EACH_ABLATION_MODE = True to print every mode.)"
        )


def _format_score_for_print(score: float) -> str:
    s = float(score or 0.0)
    if 0.0 <= s <= 1.0:
        return f"{s:.2%} (cosine-like)"
    return f"{s:.4f} (rank score)"


def _print_doc_summary(title: str, docs: List[Dict[str, Any]]) -> None:
    _vprint(title)
    if not docs:
        _vprint("  (no documents retrieved)")
        return
    for j, d in enumerate(summarize_rag_docs(docs, max_docs=5), 1):
        sc = _format_score_for_print(float(d.get("score", 0.0) or 0.0))
        _vprint(
            f"  [{j}] {d['title']} | {d['source_file']} | {sc}\n"
            f"      {d['snippet']}"
        )


_RERANK_MODE_LABELS: Dict[str, str] = {
    "none": "no rerank (vector + MMR only)",
    "crossencoder": "CrossEncoder rerank",
    "colbert": "ColBERT rerank",
    "crossencoder_colbert": "CrossEncoder + ColBERT (sum)",
}


def _attack_type_for_display(sample: Dict[str, Any]) -> str:
    return str(
        sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"
    ).strip().upper()


def _print_compact_top_sources_for_mode(
    mode_key: str,
    rag_results: List[Dict[str, Any]],
    *,
    top_n: int,
) -> None:
    label = _RERANK_MODE_LABELS.get(str(mode_key), str(mode_key))
    print(f"\nTop {int(top_n)} sources — {mode_key} ({label})")
    docs = summarize_rag_docs(
        rag_results or [],
        max_docs=int(top_n),
        snippet_chars=1,
    )
    if not docs:
        print("  (none)")
        return
    for j, d in enumerate(docs, 1):
        sf = str(d.get("source_file") or "").strip() or "(unknown source)"
        title = str(d.get("title") or "").strip() or "(no title)"
        print(f"  [{j}] {sf} | {title}")


def _print_llm_response_compact(llm_response: Optional[Dict[str, Any]]) -> None:
    print("\n--- LLM response (JSON) ---")
    if not llm_response:
        print("Failed to get LLM response")
        return
    print(json.dumps(llm_response, indent=2, ensure_ascii=False))


def _print_llm_response_summary(llm_response: Optional[Dict[str, Any]]) -> None:
    if not llm_response:
        print("Failed to get LLM response")
        return
    print("\n--- LLM response ---")
    print(f"  Threat level: {llm_response.get('threat_level')}")
    print(f"  Execution priority: {llm_response.get('execution_priority')}")
    print(f"  Primary actions: {llm_response.get('primary_actions', [])}")
    reasoning = (llm_response.get("overall_reasoning") or "")[:500]
    if len(llm_response.get("overall_reasoning") or "") > 500:
        reasoning += "..."
    print(f"  Overall reasoning: {reasoning}")


def run_agent_pipeline_for_sample(
    sample: Dict[str, Any],
    vector_store: Any,
    attack_actions_data: Optional[Dict[str, Any]],
    agentic_features_data: Optional[Dict[str, Any]],
    results_action_dir: Path,
    *,
    top_k_retrieve: int = 10,
    query_strategy: Any = "template",
    rerank_mode: str = "crossencoder_colbert",
    run_rerank_ablation: bool = False,
    comparison_modes: Optional[List[str]] = None,
    index_embed_model: Optional[str] = None,
) -> None:
    """Run retrieval → optional rerank ablation → LLM. See module flags for stdout detail."""

    if _PIPELINE_COMPACT_STDOUT:
        print("=" * 80)
        print(
            f"Sample {sample.get('sample_id')} | "
            f"attack_type={_attack_type_for_display(sample)} | "
            f"confidence={float(sample.get('confidence', 0) or 0):.1%}"
        )
    else:
        _vprint("=" * 80)
        _vprint(
            f"Sample {sample.get('sample_id')} | "
            f"label={sample.get('predicted_label', 'UNKNOWN')} | "
            f"conf={sample.get('confidence', 0):.1%}"
        )

    q_template = build_template_rag_query(sample)
    q_rephrased = rephrase_template_query_deterministic(q_template)
    q_llm_concise, q_llm_expanded = build_llm_rag_queries(sample)

    generated_queries: Dict[str, str] = {
        "template": q_template,
        "rephrase": q_rephrased,
        "llm_concise": q_llm_concise,
        "llm_expanded": q_llm_expanded,
    }

    strategy_keys = _normalize_strategy_to_keys(query_strategy)
    if strategy_keys == ["multi"]:
        strategy_keys = ["template", "rephrase"]

    strategy_keys = [k for k in strategy_keys if k in generated_queries]
    if not strategy_keys:
        strategy_keys = ["template"]

    retrieval_queries = [generated_queries[k] for k in strategy_keys]
    retrieval_queries = [q for q in retrieval_queries if q] or [generated_queries["template"]]

    if not _PIPELINE_COMPACT_STDOUT:
        _vprint(f"Query strategy: {strategy_keys} | queries={len(retrieval_queries)}")

        _vprint("\nRetrieval queries used (vector search / MMR anchor):")
        for i, sk in enumerate(strategy_keys):
            q = generated_queries.get(sk, "")
            _vprint(f"  [{i + 1}] key={sk!r}")
            _vprint(f"      {q}")

        if _PIPELINE_SHOW_UNUSED_QUERY_VARIANTS:
            _vprint("\nOther query variants (not used this run):")
            for k in ("template", "rephrase", "llm_concise", "llm_expanded"):
                if k in strategy_keys:
                    continue
                _vprint(f"\n[{k}]\n{generated_queries.get(k, '')}")
        else:
            unused = [
                k
                for k in ("template", "rephrase", "llm_concise", "llm_expanded")
                if k not in strategy_keys
            ]
            if unused:
                _vprint(f"\n(Unused query variant keys this run: {', '.join(unused)})")

    if len(retrieval_queries) == 1:
        query_for_save = retrieval_queries[0]
    else:
        parts = [f"[{i+1}] {q}" for i, q in enumerate(retrieval_queries)]
        query_for_save = "MULTI_QUERY\n\n" + "\n\n---\n\n".join(parts)

    modes = (
        list(comparison_modes)
        if (run_rerank_ablation and comparison_modes)
        else None
    )
    if run_rerank_ablation:
        modes = modes or ["none", "crossencoder", "colbert"]

    if run_rerank_ablation and modes:
        if not _PIPELINE_COMPACT_STDOUT:
            _vprint(
                f"\nRerank ablation: {len(modes)} mode(s) -> "
                f"{len(modes)} LLM call(s) for this sample."
            )
        modes_payload: Dict[str, Dict[str, Any]] = {}
        per_query_results: Optional[List[List[Dict[str, Any]]]] = None

        for mode_i, mode in enumerate(modes):
            rag_results, pq = retrieve_rag_context_multi(
                vector_store,
                retrieval_queries,
                top_k=top_k_retrieve,
                rerank_mode=str(mode),
            )
            if per_query_results is None:
                per_query_results = pq
                if (
                    not _PIPELINE_COMPACT_STDOUT
                    and _PIPELINE_PRINT_PER_QUERY_CHUNKS
                ):
                    _vprint("\nRAG retrieval results (per query, vector scores; before merge/MMR/rerank):")
                    for i, q in enumerate(retrieval_queries):
                        _vprint(f"\nQuery {i+1}/{len(retrieval_queries)}:\n{q}")
                        _print_doc_summary(
                            f"Retrieved {len(per_query_results[i])} docs (showing up to 5):",
                            per_query_results[i],
                        )

            label = _RERANK_MODE_LABELS.get(str(mode), str(mode))
            if _PIPELINE_COMPACT_STDOUT:
                _print_compact_top_sources_for_mode(
                    str(mode),
                    rag_results,
                    top_n=_PIPELINE_COMPACT_TOP_SOURCES,
                )
            else:
                _vprint("")
                _vprint("-" * 80)
                _vprint(f"RERANK MODE: {mode} — {label}")
                _vprint("-" * 80)
                if _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY:
                    _print_doc_summary(
                        f"Merged sections (title + score + snippet; full text in RAG CONTEXT below): {len(rag_results)} total (showing up to 5):",
                        rag_results,
                    )
                _vprint("")

            prompt = create_prompt(
                sample, rag_results, attack_actions_data, agentic_features_data
            )
            if not _PIPELINE_COMPACT_STDOUT:
                _print_pipeline_prompt_artifacts(
                    rag_results,
                    prompt,
                    header_extra=f" [rerank_mode={mode}]",
                    ablation_mode_index=mode_i,
                    ablation_modes_total=len(modes),
                )
            if not _PIPELINE_COMPACT_STDOUT:
                print(f"Calling LLM API (rerank_mode={mode})...")
            llm_response = call_llm_api(prompt)
            if _PIPELINE_COMPACT_STDOUT:
                _print_llm_response_compact(llm_response)
            else:
                _print_llm_response_summary(llm_response)

            modes_payload[str(mode)] = {
                "label": label,
                "rag_results": rag_results,
                "llm_response": llm_response,
            }

        retrieval_entries = [
            {"strategy_key": sk, "query_text": generated_queries[sk]}
            for sk in strategy_keys
        ]
        try:
            cmp_path = save_rerank_comparison_report(
                sample,
                query_for_save,
                modes_payload,
                results_action_dir,
                retrieval_query_entries=retrieval_entries,
                index_embed_model=index_embed_model,
            )
            print(f"Saved rerank comparison report to {cmp_path.name}")
        except Exception as e:
            print(f"Error saving rerank comparison report: {e}")

        print("=" * 80)
        return

    rag_results, per_query_results = retrieve_rag_context_multi(
        vector_store,
        retrieval_queries,
        top_k=top_k_retrieve,
        rerank_mode=rerank_mode,
    )

    if not _PIPELINE_COMPACT_STDOUT and _PIPELINE_PRINT_PER_QUERY_CHUNKS:
        _vprint("\nRAG retrieval results:")
        for i, q in enumerate(retrieval_queries):
            _vprint(f"\nQuery {i+1}/{len(retrieval_queries)}:\n{q}")
            _print_doc_summary(
                f"Retrieved {len(per_query_results[i])} docs (showing up to 5):",
                per_query_results[i],
            )

    if _PIPELINE_COMPACT_STDOUT:
        _print_compact_top_sources_for_mode(
            str(rerank_mode),
            rag_results,
            top_n=_PIPELINE_COMPACT_TOP_SOURCES,
        )
    else:
        _vprint("")
        if _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY:
            _print_doc_summary(
                f"Merged sections (title + score + snippet; full text in RAG CONTEXT below): {len(rag_results)} total (showing up to 5):",
                rag_results,
            )
        _vprint("")

    prompt = create_prompt(
        sample, rag_results, attack_actions_data, agentic_features_data
    )
    if not _PIPELINE_COMPACT_STDOUT:
        _print_pipeline_prompt_artifacts(rag_results, prompt)

    if not _PIPELINE_COMPACT_STDOUT:
        print("Calling LLM API...")
    llm_response = call_llm_api(prompt)

    if not llm_response:
        print("Failed to get LLM response")
        print("=" * 80)
        return

    if _PIPELINE_COMPACT_STDOUT:
        _print_llm_response_compact(llm_response)
    else:
        _print_llm_response_summary(llm_response)

    try:
        output_file = save_action_plan(
            sample,
            query_for_save,
            rag_results,
            llm_response,
            results_action_dir,
            variant=rerank_mode if rerank_mode != "crossencoder_colbert" else None,
        )
        print(f"Saved action plan to {output_file.name}")
    except Exception as e:
        print(f"Error saving action plan: {e}")

    print("=" * 80)

## Action pipeline driver

Run the cells below **in order** after the **retrieval / pipeline definitions** cell.

1. **Preload rerankers** — loads CrossEncoder and ColBERT once so loader noise stays out of the first sample trace.
2. **Configuration** — query strategy and rerank ablation flags (see also `_PIPELINE_*` toggles in the definitions cell for prompt/RAG verbosity).
3. **Execute pipeline** — retrieval, printed RAG context + prompts, LLM calls.


In [4]:
# Preload rerank models (optional but recommended before Execute).
ensure_rerankers_loaded()


RuntimeError: MSVC (cl.exe) not found and vcvars64.bat could not be located. Install 'Desktop development with C++' in Visual Studio 2022 or Build Tools, or start Jupyter from 'x64 Native Tools Command Prompt for VS 2022'.

In [ ]:
# -----------------------------
# Retrieval / driver configuration
# -----------------------------
# Compact notebook output: attack type, top-N source|title per rerank mode, full LLM JSON.
_PIPELINE_COMPACT_STDOUT = True
_PIPELINE_COMPACT_TOP_SOURCES = 10
if _PIPELINE_COMPACT_STDOUT:
    _PIPELINE_SHOW_UNUSED_QUERY_VARIANTS = False
    _PIPELINE_PRINT_PER_QUERY_CHUNKS = False
    _PIPELINE_PRINT_MERGED_CHUNK_SUMMARY = False
    _PIPELINE_PRINT_RAG_CONTEXT_VERBATIM = False
    _PIPELINE_PRINT_FULL_LLM_PROMPT = False

QUERY_STRATEGY = "template"  # or: ["template", "rephrase"]

RUN_RERANK_ABLATION = True
RERANK_COMPARISON_MODES = ["none", "crossencoder", "colbert"]
DEFAULT_RERANK_MODE = "crossencoder_colbert"

print("Driver configuration:")
print(f"  _PIPELINE_COMPACT_STDOUT = {_PIPELINE_COMPACT_STDOUT} (top {_PIPELINE_COMPACT_TOP_SOURCES} sources per rerank mode)")
print(f"  QUERY_STRATEGY = {QUERY_STRATEGY!r}")
print(f"  RUN_RERANK_ABLATION = {RUN_RERANK_ABLATION}")
print(f"  RERANK_COMPARISON_MODES = {RERANK_COMPARISON_MODES}")
print(f"  DEFAULT_RERANK_MODE = {DEFAULT_RERANK_MODE!r}")
print(f"  samples_for_actions = {len(samples_for_actions)} sample(s)")
print(f"  RESULTS_DIR = {RESULTS_DIR}")


In [ ]:
# Execute: RAG + LLM for each queued sample.
for sample in samples_for_actions:
    run_agent_pipeline_for_sample(
        sample,
        vector_store,
        attack_actions_data,
        agentic_features_data,
        RESULTS_DIR,
        top_k_retrieve=10,
        query_strategy=QUERY_STRATEGY,
        rerank_mode=DEFAULT_RERANK_MODE,
        run_rerank_ablation=RUN_RERANK_ABLATION,
        comparison_modes=RERANK_COMPARISON_MODES,
        index_embed_model=_INDEX_EMBED_MODEL,
    )

print("Done. Action plans in:", RESULTS_DIR)


In [ ]:
# RAG context keyword scores (nine rerank comparison JSON files)
# Requires prior cells: RESULTS_DIR, attack_actions_data, agentic_features_data (may be None).
# Picks the newest rerank_comparison_sample_<id>_*.json per sample_id 0..8.
# Writes CSV with one row per (sample, doc_rank). Same rank index can refer to different
# chunks per rerank mode, so we do not store document titles.
# Per mode (none / crossencoder / colbert): attack_terms_hit_rate, mitigation_terms_hit_rate,
# agentic_hit_rate, then the rerank column = sum of those three for that mode's chunk at that rank.

from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

_RERANK_COLS = {
    "none": "no rerank",
    "crossencoder": "with rerank",
    "colbert": "with rerank colbert",
}
_MODE_ORDER = ("none", "crossencoder", "colbert")
# CSV column prefixes for per-mode metrics (sum columns use _RERANK_COLS values).
_MODE_PREFIX = {
    "none": "none",
    "crossencoder": "crossencoder",
    "colbert": "colbert",
}

_LABEL_ALIASES: Dict[str, List[str]] = {
    "BENIGN": ["benign", "normal traffic", "non-malicious", "legitimate"],
    "DDOS": ["ddos", "distributed denial", "denial of service"],
    "DOS": ["denial of service", "flooding", "flood", "dos attack"],
    "BOT": ["botnet", "automated", "command and control"],
    "PORTSCAN": ["port scan", "portscan", "reconnaissance", "scanning"],
    "WEBATTACK": ["web attack", "injection", "waf", "xss"],
    "SSHPATATOR": ["ssh", "brute force", "credential"],
    "FTPPATATOR": ["ftp", "brute force", "credential"],
    "OTHERS": ["intrusion", "malicious", "anomaly"],
}


def _norm(s: str) -> str:
    return unicodedata.normalize("NFKC", s or "").lower()


def _pick_latest_per_sample_id(paths: List[Path]) -> Dict[int, Path]:
    best: Dict[int, Tuple[float, Path]] = {}
    pat = re.compile(r"rerank_comparison_sample_(\d+)_")
    for p in paths:
        m = pat.search(p.name)
        if not m:
            continue
        sid = int(m.group(1))
        mt = p.stat().st_mtime
        prev = best.get(sid)
        if prev is None or mt > prev[0]:
            best[sid] = (mt, p)
    return {k: v[1] for k, v in best.items()}


def _attack_terms(label: str) -> List[str]:
    u = (label or "UNKNOWN").strip().upper()
    parts = {_norm(u), _norm(u.replace("_", " "))}
    for a in _LABEL_ALIASES.get(u, []):
        parts.add(_norm(a))
    return sorted(p for p in parts if p and len(p) >= 2)


def _mitigation_phrases(attack_options: Dict[str, Any], label: str) -> List[str]:
    u = (label or "UNKNOWN").strip().upper()
    attacks = (attack_options or {}).get("attacks") or {}
    acts = attacks.get(u) or []
    out: List[str] = []
    for a in acts:
        s = _norm(str(a))
        if len(s) >= 3:
            out.append(s)
    return out


def _agentic_capability_phrases(agentic: Optional[Dict[str, Any]]) -> List[str]:
    if not agentic or not isinstance(agentic, dict):
        return []
    agents = agentic.get("agents")
    if not isinstance(agents, dict):
        return []
    out: List[str] = []
    for tier in ("RAN", "Edge", "Core"):
        block = agents.get(tier)
        if not isinstance(block, dict):
            continue
        for c in block.get("action_capabilities") or []:
            nc = _norm(str(c))
            if len(nc) >= 3:
                out.append(nc)
    return list(dict.fromkeys(out))


def _fraction_substrings(blob: str, phrases: List[str]) -> float:
    if not phrases:
        return 0.0
    hits = sum(1 for ph in phrases if ph and ph in blob)
    return hits / len(phrases)


def _fraction_attack_hits(blob: str, tokens: List[str]) -> float:
    if not tokens:
        return 0.0
    hits = 0
    for t in tokens:
        if not t:
            continue
        if len(t) <= 5:
            if re.search(rf"(?<![a-z0-9]){re.escape(t)}(?![a-z0-9])", blob):
                hits += 1
        elif t in blob:
            hits += 1
    return hits / len(tokens)


def _agentic_tier_coverage(blob: str) -> float:
    found = 0
    for tier, pat in (
        ("ran", r"(?<![a-z0-9_])ran(?![a-z0-9_])"),
        ("edge", r"(?<![a-z0-9_])edge(?![a-z0-9_])"),
        ("core", r"(?<![a-z0-9_])core(?![a-z0-9_])"),
    ):
        if re.search(pat, blob):
            found += 1
    return found / 3.0


def _rag_keyword_parts(
    title: str,
    text: str,
    *,
    attack_tokens: List[str],
    mitigation_phrases: List[str],
    agentic_phrases: List[str],
) -> Tuple[float, float, float, float]:
    """
    Returns (attack_hit_rate, mitigation_hit_rate, agentic_hit_rate, total_sum).

    Mitigation uses allowed-action phrases only (not max-blended with agentic capabilities).
    Agentic is max(capability-phrase coverage, RAN/Edge/Core tier mention coverage).
    """
    blob = _norm(f"{title}\n{text}")
    a = _fraction_attack_hits(blob, attack_tokens)
    mit = _fraction_substrings(blob, mitigation_phrases) if mitigation_phrases else 0.0
    cap = _fraction_substrings(blob, agentic_phrases) if agentic_phrases else 0.0
    tier = _agentic_tier_coverage(blob)
    ag = max(cap, tier)
    total = a + mit + ag
    return a, mit, ag, total


def load_nine_rerank_reports(
    action_dir: Path,
    *,
    sample_ids: Optional[List[int]] = None,
) -> List[Tuple[int, Path]]:
    action_dir = Path(action_dir)
    paths = list(action_dir.glob("rerank_comparison_sample_*.json"))
    by_sid = _pick_latest_per_sample_id(paths)
    ids = sample_ids if sample_ids is not None else list(range(9))
    out: List[Tuple[int, Path]] = []
    for sid in ids:
        p = by_sid.get(sid)
        if p is None:
            raise FileNotFoundError(
                f"No rerank_comparison_sample_{sid}_*.json under {action_dir}"
            )
        out.append((sid, p))
    return sorted(out, key=lambda x: x[0])


def build_rag_scores_dataframe(
    report_paths: List[Tuple[int, Path]],
    attack_options: Optional[Dict[str, Any]],
    agentic: Optional[Dict[str, Any]],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    agentic_phrases = _agentic_capability_phrases(agentic)

    for sid, path in report_paths:
        with open(path, encoding="utf-8") as f:
            rep = json.load(f)
        label = str((rep.get("prediction") or {}).get("predicted_label") or "UNKNOWN")
        ulabel = label.strip().upper()
        atk = _attack_terms(ulabel)
        mit = _mitigation_phrases(attack_options or {}, ulabel)

        modes_data = rep.get("modes") or {}
        for mk in _MODE_ORDER:
            if mk not in modes_data:
                raise ValueError(f"{path.name}: missing mode {mk!r} (expected ablation JSON)")

        per_mode: Dict[str, List[Dict[str, Any]]] = {}
        for mk in _MODE_ORDER:
            top = ((modes_data[mk].get("rag_info") or {}).get("top_results")) or []
            per_mode[mk] = top

        n_docs = max(len(per_mode[mk]) for mk in _MODE_ORDER)

        for di in range(n_docs):
            row: Dict[str, Any] = {
                "attack_group": f"sample_{sid}_{ulabel}",
                "sample_id": sid,
                "predicted_label": ulabel,
                "doc_rank": di + 1,
            }
            for mk in _MODE_ORDER:
                docs = per_mode[mk]
                prefix = _MODE_PREFIX[mk]
                ca = f"{prefix}_attack_terms_hit_rate"
                cm = f"{prefix}_mitigation_terms_hit_rate"
                cg = f"{prefix}_agentic_hit_rate"
                csum = _RERANK_COLS[mk]
                if di >= len(docs):
                    row[ca] = row[cm] = row[cg] = row[csum] = None
                    continue
                d = docs[di]
                title = str(d.get("title") or "")
                body = str(d.get("text") or "")
                a, mit, ag, total = _rag_keyword_parts(
                    title,
                    body,
                    attack_tokens=atk,
                    mitigation_phrases=mit,
                    agentic_phrases=agentic_phrases,
                )
                row[ca] = round(a, 4)
                row[cm] = round(mit, 4)
                row[cg] = round(ag, 4)
                row[csum] = round(total, 4)
            rows.append(row)

    col_order = [
        "attack_group",
        "sample_id",
        "predicted_label",
        "doc_rank",
    ]
    for mk in _MODE_ORDER:
        p = _MODE_PREFIX[mk]
        col_order.extend(
            [
                f"{p}_attack_terms_hit_rate",
                f"{p}_mitigation_terms_hit_rate",
                f"{p}_agentic_hit_rate",
                _RERANK_COLS[mk],
            ]
        )
    df = pd.DataFrame(rows)
    return df[[c for c in col_order if c in df.columns]]


_action_dir = Path(RESULTS_DIR)
_reports = load_nine_rerank_reports(_action_dir)
_df_scores = build_rag_scores_dataframe(
    _reports,
    attack_actions_data,
    agentic_features_data if isinstance(agentic_features_data, dict) else None,
)
_csv_out = _action_dir / "rag_context_rerank_keyword_scores.csv"
_df_scores.to_csv(_csv_out, index=False)
print(f"Wrote {_csv_out} ({len(_df_scores)} rows).")
try:
    from IPython.display import display

    display(_df_scores)
except Exception:
    print(_df_scores.to_string())


## Rerank report metrics (optional)

After generating `rerank_comparison_sample_*.json`, run the next cell (needs `vector_store` from the load cell).

The next cell reads reports from **`RERANK_METRICS_REPORT_DIR`** (default `RAG_docs/sample_for_analysis`). Set it to `RESULTS_DIR` if you want metrics on freshly written `action_plans/` exports instead.

New reports include `retrieval_queries`, `mmr_rerank_anchor_query`, `index_embed_model`, and per-chunk `parent_id` / `child_index` / `source_file` under `modes.*.rag_info.top_results`.

In [ ]:
# Optional: mean cosine(query, top-k chunks) + pairwise LLM field comparison per report.
try:
    from utils.rag_utils import embeddings_from_vector_store
except ImportError:
    # Older rag_utils or stale kernel: same logic as utils.rag_utils.embeddings_from_vector_store
    def embeddings_from_vector_store(vector_store):
        for attr in ("embedding_function", "embeddings"):
            e = getattr(vector_store, attr, None)
            if e is not None and hasattr(e, "embed_query") and hasattr(e, "embed_documents"):
                return e
        raise ValueError(
            "Vector store has no LangChain-compatible Embeddings instance "
            "(expected embedding_function or embeddings with embed_query/embed_documents)."
        )

from utils.rerank_metrics import downstream_comparison_for_report, embedding_metrics_for_report

# Directory containing rerank_comparison_sample_*.json (default: curated analysis copies).
RERANK_METRICS_REPORT_DIR = Path("RAG_docs/sample_for_analysis")
# RERANK_METRICS_REPORT_DIR = RESULTS_DIR  # use this for reports under action_plans/

_MET_TOPK = 10
_emb = embeddings_from_vector_store(vector_store)
_paths = sorted(RERANK_METRICS_REPORT_DIR.glob("rerank_comparison_sample_*.json"))
if not _paths:
    print("No rerank_comparison_sample_*.json in", RERANK_METRICS_REPORT_DIR.resolve())
else:
    _cos_acc: dict[str, list[float]] = {}
    for _p in _paths[:20]:
        import json as _json

        _rep = _json.loads(_p.read_text(encoding="utf-8"))
        if _rep.get("report_type") != "rerank_ablation_comparison":
            continue
        _e = embedding_metrics_for_report(_rep, _emb, top_k=_MET_TOPK)
        _d = downstream_comparison_for_report(_rep)
        print(f"\n{_p.name} | sample_id={_rep.get('sample_id')}")
        for _mk, _st in (_e.get("per_mode") or {}).items():
            v = float(_st.get("mean_top_k_cosine_query_chunk", 0.0))
            _cos_acc.setdefault(_mk, []).append(v)
            print(f"  cosine_mean_top{_MET_TOPK} [{_mk}]: {v:.4f}")
        for _row in (_d.get("pairwise") or [])[:3]:
            print("  pair:", _row)
    print("\n--- aggregate (up to 20 files) ---")
    for _mk, vs in sorted(_cos_acc.items()):
        print(f"  [{_mk}] mean {sum(vs)/len(vs):.4f} over {len(vs)} reports")

### LLM vs reference (Score.ipynb-style) — 9 × 3 rerank matrix

Uses `build_reference_text_for_label` + `overall_reasoning` per rerank mode on `RAG_docs/sample_for_analysis`. Rows = `sample_id` 0–8 with predicted label; columns = the three rerank methods.

**At the very end of the cell** (after the heatmaps):

- **TABLE 1** — For each rerank method, the **mean** of BERTScore F1, SBERT cosine, and ROUGE-1 **averaged over all 9 attacks** (same as taking column means of the 9×3 matrices).
- **TABLE 2** — For each of the **9 attack types**, which rerank method has the **highest BERTScore F1** on that row (tie → first column).

Needs prior **imports** (`Path`) and loads SBERT/BERTScore here (can take ~30–60s first run).

In [ ]:
# Score.ipynb-style: reference text vs overall_reasoning → 9×3 matrices (sample × rerank mode)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)

from sentence_transformers import SentenceTransformer
from utils.rerank_plan_scoring import (
    build_reference_text_for_label,
    compute_lexical_similarity_scores,
    extract_overall_reasoning,
    load_rerank_comparison_files,
    load_scoring_kb,
)

_RDIR = Path("RAG_docs/sample_for_analysis")
_MODE_ORDER = ["none", "crossencoder", "colbert"]
_MODE_COL_NAMES = ["no rerank", "crossencoder", "colbert"]

_attack_kb, _agents_kb = load_scoring_kb()
_reps = load_rerank_comparison_files(_RDIR)
print(f"Loaded {len(_reps)} rerank_comparison report(s) from {_RDIR}")

print("Loading SentenceTransformer('all-MiniLM-L6-v2') …")
_sbert = SentenceTransformer("all-MiniLM-L6-v2")

_label_by_sid: dict[int, str] = {}
_mat_f1 = np.full((9, 3), np.nan, dtype=float)
_mat_sb = np.full((9, 3), np.nan, dtype=float)
_mat_r1 = np.full((9, 3), np.nan, dtype=float)

for _rep in _reps:
    sid = int(_rep.get("sample_id", -1))
    if sid < 0 or sid > 8:
        print(f"  skip sample_id={sid} (expected 0..8)")
        continue
    plab = str((_rep.get("prediction") or {}).get("predicted_label") or "OTHERS")
    _label_by_sid[sid] = plab
    ref = build_reference_text_for_label(plab, _attack_kb, _agents_kb)
    modes = _rep.get("modes") or {}
    for j, mk in enumerate(_MODE_ORDER):
        block = modes.get(mk)
        if not isinstance(block, dict):
            continue
        plan = block.get("llm_action_plan")
        txt = extract_overall_reasoning(plan if isinstance(plan, dict) else None)
        if not txt:
            continue
        sc = compute_lexical_similarity_scores(ref, txt, _sbert)
        _mat_f1[sid, j] = sc["bertscore_f1"]
        _mat_sb[sid, j] = sc["sbert_cosine"]
        _mat_r1[sid, j] = sc["rouge1"]

_yt = [f"id {i} | {_label_by_sid.get(i, '?')}" for i in range(9)]
df_bertscore_f1 = pd.DataFrame(_mat_f1, index=_yt, columns=_MODE_COL_NAMES)
df_sbert_cos = pd.DataFrame(_mat_sb, index=_yt, columns=_MODE_COL_NAMES)
df_rouge1 = pd.DataFrame(_mat_r1, index=_yt, columns=_MODE_COL_NAMES)

print("\n=== BERTScore F1 (9 × 3) ===")
print(df_bertscore_f1.to_string())
print("\n=== SBERT cosine (9 × 3) ===")
print(df_sbert_cos.to_string())
print("\n=== ROUGE-1 F1 (9 × 3) ===")
print(df_rouge1.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
sns.heatmap(df_bertscore_f1, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="viridis", ax=axes[0])
axes[0].set_title("BERTScore F1")
sns.heatmap(df_sbert_cos, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="mako", ax=axes[1])
axes[1].set_title("SBERT cosine")
sns.heatmap(df_rouge1, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="rocket_r", ax=axes[2])
axes[2].set_title("ROUGE-1")
plt.tight_layout()
plt.show()

# --- Final summary (mean across 9 attacks; best ranker per attack) ---
tbl_mean_by_ranker = pd.DataFrame(
    {
        "mean_bertscore_f1": df_bertscore_f1.mean(skipna=True),
        "mean_sbert_cosine": df_sbert_cos.mean(skipna=True),
        "mean_rouge1": df_rouge1.mean(skipna=True),
    }
)
tbl_mean_by_ranker.index.name = "rerank_method"

_rows_best = []
for _i in range(9):
    _row = df_bertscore_f1.iloc[_i]
    _atk = _label_by_sid.get(_i, "?")
    if _row.notna().sum() == 0:
        _rows_best.append((_i, _atk, "—", float("nan")))
    else:
        _w = _row.idxmax(skipna=True)
        _rows_best.append((_i, _atk, str(_w), float(_row[_w])))

tbl_best_ranker_per_attack = pd.DataFrame(
    _rows_best,
    columns=[
        "sample_id",
        "attack_type",
        "best_rerank_method",
        "bertscore_f1_at_best",
    ],
)

print("\n" + "=" * 76)
print("TABLE 1 — Mean score across ALL 9 attacks (per rerank method)")
print("=" * 76)
print(tbl_mean_by_ranker.to_string())
print("\n" + "=" * 76)
print("TABLE 2 — Best rerank method per attack (highest BERTScore F1 in that row)")
print("=" * 76)
print(tbl_best_ranker_per_attack.to_string())

try:
    from IPython.display import display

    display(tbl_mean_by_ranker)
    display(tbl_best_ranker_per_attack)
except Exception:
    pass